# <font color='red' size='6'>Inteligência Artificial</font>

## Projeto da Disciplina

### Aplicação de IA para Predição do Risco de AVC – PNS 2019

#### Notebook 1 – Obtenção e preparação dos dados
##### <p><center><b>Grupo</b></center></p>

Marcar com um X a turma a qual o grupo pertence.</br>

<b>Turma: ( &nbsp;&nbsp;&nbsp;&nbsp; )   7G</br> Turma: ( X ) 7N </b><p>
<table width="450" border="2">
    <tr>
        <td><b>Nome do Aluno</b> </td>
        <td><b>RA</b></td>
     </tr>
   <tr>
       <td>Luana Miho Yasuda</td>
       <td>10396439</td>
   </tr>
   <tr>
       <td>Luis Fernando de Mesquita Pereira</td>
       <td>10410686</td>
   </tr>
   <tr>
       <td>Richard Barbosa Sanches</td>
       <td>10420179</td>
   </tr>
   <tr>
       <td>Totti Ferreira Gomes</td>
       <td>10428490</td>
   </tr>
   <tr>
       <td>Gabriel Resende Menezes</td>
       <td>10419003</td>
   </tr>
</table>

## Definindo o problema

O objetivo do projeto é classificar se uma pessoa tem ou não diagnóstico médico de AVC a partir de características
demográficas, clínicas, comportamentais e socioeconômicas, usando os microdados da Pesquisa Nacional de Saúde
(PNS) 2019 do IBGE. É uma classificação binária com classes bem desbalanceadas, já que o AVC é um evento raro.

Este primeiro notebook cuida da parte de dados: baixar os microdados, ler o arquivo, selecionar as variáveis que
vamos usar (Tabela 1 do artigo), aplicar o recorte de 40 anos ou mais e gerar a base que o notebook 2 (análise
exploratória) e, depois, a modelagem vão usar.

Os microdados não ficam no repositório. Rodando este notebook eles são baixados para `data/raw/` e a base gerada
vai para `data/processed/`.

In [2]:
import sys
sys.path.append("..")  # para importar a pasta src

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src import pns
from src.pesos import prevalencia

pns.DIR_TAB.mkdir(parents=True, exist_ok=True)

## Download dos microdados

São dois arquivos zip do FTP do IBGE (uns 30 MB no total): os microdados (`PNS_2019.txt`, arquivo de largura fixa
com ~455 MB descompactado) e a documentação (dicionário em xls e o programa de leitura `input_PNS_2019.sas`). Se já
estiverem em `data/raw/`, o download é pulado.

In [3]:
arquivos = pns.baixar_pns2019()
for nome, caminho in arquivos.items():
    print(f"{nome}: data/raw/{caminho.name}")

PNS_2019_20220525.zip já existe, pulando download
Dicionario_e_input_20220530.zip já existe, pulando download
txt: data/raw/PNS_2019.txt
sas: data/raw/input_PNS_2019.sas
dicionario: data/raw/dicionario_PNS_microdados_2019.xls


## Leitura do arquivo

O arquivo do IBGE é de largura fixa, ou seja, cada variável ocupa posições fixas da linha. Em vez de digitar as
posições na mão, lemos o programa SAS que o próprio IBGE distribui e pegamos o início e o tamanho de cada
variável de lá.

In [4]:
layout = pns.ler_layout(arquivos["sas"])
print("variáveis no layout completo:", len(layout))

layout_sel = layout.set_index("variavel").loc[list(pns.VARIAVEIS)].reset_index()
layout_sel["descricao"] = layout_sel["variavel"].map(pns.VARIAVEIS)
layout_sel

variáveis no layout completo: 1088


,variavel,inicio,largura,texto,descricao
0,V0001,1,2,True,Unidade da Federação
1,V0024,3,7,True,Estrato
2,UPA_PNS,10,9,True,Unidade Primária de Amostragem
3,V0026,31,1,True,"Situação censitária (1 urbano, 2 rural)"
4,M001,527,1,True,Entrevista do adulto selecionado (1 = realizada)
5,V00291,1426,14,False,Peso do morador selecionado com calibração
6,C006,108,1,True,Sexo
7,C008,117,3,False,Idade (anos)
8,C009,120,1,True,Cor ou raça
9,Q00201,807,1,True,Diagnóstico médico de hipertensão


In [5]:
bruto = pns.extrair_variaveis(arquivos["txt"], layout, pns.VARIAVEIS)
pns.DIR_PROC.mkdir(parents=True, exist_ok=True)
bruto.to_parquet(pns.ARQ_EXTRATO, index=False)
print(f"{len(bruto):,} moradores e {bruto.shape[1]} variáveis")
bruto.head()

293,726 moradores e 22 variáveis


,V0001,V0024,UPA_PNS,V0026,M001,V00291,C006,C008,C009,Q00201,Q03001,P00104,P00404,P050,P052,P027,P034,VDD004A,VDF003,VDF004,I00102,Q068
0,11,1110011,110000016,1,1,00368.87516575,2,055,1,1,2,060.0,145,3,1,2,1,2,00000350,2,2,2
1,11,1110011,110000016,1,NaN,NaN,1,069,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,00000350,2,2,NaN
2,11,1110011,110000016,1,NaN,NaN,1,031,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,00000350,2,2,NaN
3,11,1110011,110000016,1,NaN,NaN,1,009,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,00000350,2,2,NaN
4,11,1110011,110000016,1,NaN,NaN,2,006,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,00000350,2,2,NaN


Algumas perguntas do questionário só são feitas para quem respondeu "sim" para AVC (idade no primeiro diagnóstico,
cuidados recebidos e grau de limitação). Se entrassem como atributos, o modelo descobriria a resposta por elas.
Por isso ficaram de fora:

In [6]:
pns.VARIAVEIS_VAZAMENTO

['Q070', 'Q07208', 'Q07209', 'Q07210', 'Q07211', 'Q07212', 'Q07213', 'Q073']

## Conferindo a leitura

Para ter certeza de que o arquivo foi lido certo, calculamos a prevalência ponderada de AVC entre adultos de 18
anos ou mais (com o peso `V00291`) e comparamos com o número que o IBGE divulgou para a PNS 2019: cerca de 3,1
milhões de pessoas, por volta de 2% dos adultos.

In [7]:
m18 = (pns.num(bruto["M001"]) == 1) & (pns.num(bruto["C008"]) >= 18) & pns.num(bruto["Q068"]).isin([1, 2])
adultos = pd.DataFrame({
    "avc": (pns.num(bruto.loc[m18, "Q068"]) == 1).astype(int),
    "peso_amostral": pns.num(bruto.loc[m18, "V00291"]),
    "estrato": bruto.loc[m18, "V0024"],
    "upa": bruto.loc[m18, "UPA_PNS"],
})
conf = prevalencia(adultos)
conf["pessoas_com_avc_milhoes"] = conf["populacao_estimada"] * conf["prevalencia_%"] / 100 / 1e6
conf

,categoria,n,casos,populacao_estimada,prevalencia_%,ic95_inf_%,ic95_sup_%,erro_padrao_%,pessoas_com_avc_milhoes
0,Total,88531,1974,"159,171,311.01",1.96,1.81,2.12,0.08,3.12


## Recorte da amostra

Ficam na base os adultos que responderam ao questionário individual (`M001 = 1`), com 40 anos ou mais
(`C008 >= 40`) e com resposta sim/não na pergunta sobre AVC (`Q068`).

In [8]:
fluxo = pns.fluxo_amostra(bruto)
fluxo.to_csv(pns.DIR_TAB / "fluxo_amostra.csv", index=False)
fluxo

,etapa,registros
0,Moradores no arquivo,293726
1,Com entrevista individual realizada (M001 = 1),90846
2,Com 40 anos ou mais,54987
3,Com resposta válida sobre AVC (Q068 = 1 ou 2),54987


## Respostas "ignorado" e brancos

No dicionário do IBGE, o código 9 (999 na altura) significa "ignorado" (não sabe / não respondeu). Já os brancos
são perguntas que não foram feitas por causa do fluxo do questionário:

- `Q00201` e `Q03001` (hipertensão e diabetes): quem nunca mediu a pressão ou a glicemia não responde se tem
  diagnóstico. Tratamos esses casos como "não tem diagnóstico".
- `P052` (fumou no passado): não é perguntado para quem fuma hoje.

In [9]:
ignorados = pns.contar_ignorados(bruto)
ignorados.to_csv(pns.DIR_TAB / "ignorados_e_brancos.csv", index=False)
ignorados

,variavel,descricao,ignorado_n,branco_n,ignorado_%,branco_%
0,V0026,"Situação censitária (1 urbano, 2 rural)",0,0,0.00,0.00
1,C006,Sexo,0,0,0.00,0.00
2,C009,Cor ou raça,6,0,0.01,0.00
3,Q00201,Diagnóstico médico de hipertensão,0,538,0.00,0.98
4,Q03001,Diagnóstico médico de diabetes,0,2605,0.00,4.74
5,P00404,Altura final (cm),0,68,0.00,0.12
6,P050,Fuma atualmente,0,0,0.00,0.00
7,P052,Fumou no passado,0,7461,0.00,13.57
8,P027,Frequência de consumo de álcool,0,0,0.00,0.00
9,P034,Praticou exercício físico nos últimos 3 meses,0,0,0.00,0.00


## Base analítica

Recodificação feita em `src/pns.py` (função `construir_base`):

| Atributo | Origem | O que foi feito |
|---|---|---|
| `avc` (alvo) | Q068 | 1 = sim, 0 = não |
| `idade`, `faixa_etaria` | C008 | anos e faixas de 10 anos |
| `sexo`, `cor_raca` | C006, C009 | rótulos do dicionário (cor/raça 9 = ignorado vira NaN) |
| `regiao`, `uf` | V0001 | primeiro dígito da UF dá a região |
| `situacao` | V0026 | urbana / rural |
| `hipertensao`, `diabetes` | Q00201, Q03001 | 1/0, branco (nunca mediu) vira 0 |
| `peso`, `altura`, `imc`, `imc_categoria` | P00104, P00404 | IMC = peso / altura²; fora de 12–70 vira NaN |
| `tabagismo` | P050, P052 | fumante atual / ex-fumante / nunca fumou |
| `alcool` | P027 | não bebe / menos de 1x por mês / 1x por mês ou mais |
| `atividade_fisica` | P034 | 1/0 |
| `escolaridade`, `escolaridade_grupo` | VDD004A | de 1 a 7 e em 4 grupos |
| `renda_per_capita`, `faixa_renda` | VDF003, VDF004 | R$ e faixas de salário mínimo |
| `plano_saude` | I00102 | 1/0 |
| `peso_amostral`, `estrato`, `upa` | V00291, V0024, UPA_PNS | só para as estimativas ponderadas |

Imputação, one-hot e padronização ficam para a etapa de modelagem, dentro do pipeline, para não vazar informação
do teste para o treino.

In [10]:
base = pns.construir_base(bruto)
base.to_parquet(pns.ARQ_BASE, index=False)
print(f"{base.shape[0]:,} registros e {base.shape[1]} colunas salvos em data/processed/{pns.ARQ_BASE.name}")
base.info()

54,987 registros e 25 colunas salvos em data/processed/pns2019_avc_40mais.parquet
<class 'pandas.DataFrame'>
RangeIndex: 54987 entries, 0 to 54986
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   avc                 54987 non-null  int64   
 1   idade               54987 non-null  int64   
 2   faixa_etaria        54987 non-null  category
 3   sexo                54987 non-null  str     
 4   cor_raca            54981 non-null  str     
 5   uf                  54987 non-null  str     
 6   regiao              54987 non-null  category
 7   situacao            54987 non-null  str     
 8   hipertensao         54987 non-null  float64 
 9   diabetes            54987 non-null  float64 
 10  peso                54919 non-null  float64 
 11  altura              54919 non-null  float64 
 12  imc                 54918 non-null  float64 
 13  imc_categoria       54918 non-null  category
 14  tabagismo      

In [11]:
# resumo das colunas da base
resumo = pd.DataFrame({
    "coluna": base.columns,
    "tipo": [str(t) if str(t) != "category" else "categórica" for t in base.dtypes],
    "ausentes_%": (100 * base.isna().mean()).round(2).to_numpy(),
    "valores_distintos": base.nunique().to_numpy(),
})
resumo.to_csv(pns.DIR_TAB / "colunas_base_analitica.csv", index=False)
resumo

,coluna,tipo,ausentes_%,valores_distintos
0,avc,int64,0.00,2
1,idade,int64,0.00,66
2,faixa_etaria,categórica,0.00,5
3,sexo,str,0.00,2
4,cor_raca,str,0.01,5
5,uf,str,0.00,27
6,regiao,categórica,0.00,5
7,situacao,str,0.00,2
8,hipertensao,float64,0.00,2
9,diabetes,float64,0.00,2


In [12]:
base.drop(columns=pns.COLUNAS_DESENHO).sample(5, random_state=42)

,avc,idade,faixa_etaria,sexo,cor_raca,uf,regiao,situacao,hipertensao,diabetes,peso,altura,imc,imc_categoria,tabagismo,alcool,atividade_fisica,escolaridade,escolaridade_grupo,renda_per_capita,faixa_renda,plano_saude
24991,0,49,40-49,Masculino,Preta,28,Nordeste,Urbana,0.00,0.00,63.00,173.00,21.05,Eutrófico,Fumante atual,1x/mês ou mais,0.00,2,Sem instrução/Fund. incompleto,700.00,1/2 a 1 SM,0.00
53142,1,59,50-59,Feminino,Parda,52,Centro-Oeste,Urbana,0.00,0.00,68.00,156.00,27.94,Sobrepeso,Ex-fumante,1x/mês ou mais,1.00,2,Sem instrução/Fund. incompleto,800.00,1/2 a 1 SM,0.00
20036,0,44,40-49,Masculino,Parda,25,Nordeste,Rural,0.00,0.00,58.00,160.00,22.66,Eutrófico,Ex-fumante,1x/mês ou mais,0.00,1,Sem instrução/Fund. incompleto,133.00,Até 1/4 SM,0.00
34494,0,63,60-69,Masculino,Parda,33,Sudeste,Urbana,0.00,0.00,66.00,161.00,25.46,Sobrepeso,Nunca fumou,Não bebe,0.00,5,Médio completo/Sup. incompleto,717.00,1/2 a 1 SM,1.00
1281,0,44,40-49,Masculino,Parda,12,Norte,Urbana,0.00,0.00,85.00,175.00,27.76,Sobrepeso,Nunca fumou,Menos de 1x/mês,1.00,5,Médio completo/Sup. incompleto,750.00,1/2 a 1 SM,0.00
